# Volleyball Attack Strategy Discovery: Exploratory Data Analysis
**Course**: CMPE 257 Machine Learning - Final Project  
**Author**: Raj Dharmendra Kadvani (Student ID: 018283798)  
**Institution**: San Jose State University  
**Reference**: Van Haaren et al., *Analyzing Volleyball Match Data from the 2014 World Championships Using Machine Learning Techniques*, ACM SIGKDD 2016  
**Dataset**: VREN Volleyball Rally Dataset (haotianxia/VREN)

---
## 1. Overview and Objectives
This notebook executes **Phase 1** of our project proposal: data preparation, validation, and exploratory data analysis of volleyball rallies. We investigate the relationship between pass quality, setter choices, attack locations, opposing blocker configurations, and resulting attack success rates.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure visualization aesthetics
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.figsize'] = (10, 6)

# Add parent directory to path to import src modules
sys.path.append(os.path.abspath('..'))
from src.data_loader import load_rally_data
print('Libraries and custom modules loaded successfully.')

## 2. Load and Inspect Rally Dataset
We load the structured rally dataset using `src.data_loader.load_rally_data()`. The dataset contains sequence records across matches.

In [ ]:
df = load_rally_data()
print(f'Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

In [ ]:
print('Dataset Info:')
df.info()
print('\nMissing values count:')
print(df.isnull().sum())

## 3. Target Distribution: Attack Success Rate
The target variable `attack_success` indicates whether the attacking possession resulted in a point won (1) or was defended / lost (0).

In [ ]:
target_pct = df['attack_success'].value_counts(normalize=True) * 100
print('Target Distribution (%):')
print(target_pct)

fig, ax = plt.subplots(figsize=(7, 4.5))
sns.countplot(data=df, x='attack_success', palette=['#EF4444', '#10B981'], ax=ax)
ax.set_title('Distribution of Attack Outcomes (0 = Defended, 1 = Scored Kill)')
ax.set_xticklabels(['Defended / Lost Point', 'Attack Point Won (Kill)'])
ax.set_ylabel('Number of Possessions')
plt.tight_layout()
plt.show()

## 4. Tactical Analysis: Pass Quality Impact
A fundamental premise in volleyball theory (and Van Haaren et al. 2016) is that reception pass quality dictates offensive options.
- Rating 1: Poor / Out-of-system
- Rating 2: Medium
- Rating 3: Perfect / In-system

In [ ]:
pass_summary = df.groupby('pass_rating')['attack_success'].agg(['count', 'mean']).reset_index()
pass_summary['kill_rate_pct'] = pass_summary['mean'] * 100
print(pass_summary)

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(data=pass_summary, x='pass_rating', y='kill_rate_pct', palette='Blues_d', ax=ax)
ax.set_title('Attack Success Rate (%) by Pass Rating')
ax.set_xticklabels(['1 - Poor (Out of System)', '2 - Medium', '3 - Perfect (In-System)'])
ax.set_ylabel('Kill Rate (%)')
ax.set_ylim(0, 100)
for p in ax.patches:
    height = p.get_height()
    ax.annotate(f'{height:.1f}%', (p.get_x() + p.get_width() / 2., height + 2), ha='center')
plt.tight_layout()
plt.show()

## 5. Defensive Pressure: Impact of Opposing Blockers
How does the number of blockers contesting the hit affect scoring probability?

In [ ]:
block_summary = df.groupby('num_blockers')['attack_success'].agg(['count', 'mean']).reset_index()
block_summary['kill_rate_pct'] = block_summary['mean'] * 100
print(block_summary)

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(data=block_summary, x='num_blockers', y='kill_rate_pct', palette='Reds_r', ax=ax)
ax.set_title('Attack Success Rate (%) vs. Number of Opposing Blockers')
ax.set_xlabel('Opposing Blockers Contesting Attack')
ax.set_ylabel('Kill Rate (%)')
ax.set_ylim(0, 100)
for p in ax.patches:
    height = p.get_height()
    ax.annotate(f'{height:.1f}%', (p.get_x() + p.get_width() / 2., height + 2), ha='center')
plt.tight_layout()
plt.show()

## 6. Interaction Analysis: Set Type & Attack Efficiency Across Court Zones

In [ ]:
set_zone_matrix = pd.crosstab(df['set_type'], df['attack_zone'], values=df['attack_success'], aggfunc='mean') * 100
plt.figure(figsize=(10, 5.5))
sns.heatmap(set_zone_matrix, annot=True, fmt='.1f', cmap='YlGnBu', cbar_kws={'label': 'Kill Rate (%)'})
plt.title('Attack Success Rate (%) by Set Type and Court Zone')
plt.ylabel('Set Type')
plt.xlabel('Court Zone')
plt.tight_layout()
plt.show()

## 7. Match Summary and Leakage Verification
We inspect match distribution to verify that grouped cross-validation can be conducted without intra-match data leakage.

In [ ]:
match_counts = df.groupby('match_id')['attack_success'].agg(['count', 'mean']).reset_index()
match_counts.columns = ['Match ID', 'Rallies', 'Win Rate']
print(match_counts)
print('\nEDA findings confirmed: Strong non-linear interaction between pass rating, tempo, and blocker configuration.')